# Filtered PathVQA VLM-as-Judge Runner

Runs InternVL3.5-38B and Qwen3-VL-32B-Thinking as judges over the filtered-PathVQA
subset (`pathvqa_part{1-5}.csv`), using the rubrics defined in
`data_evaluation/vlm/prompts/benchmarks.py`.

Unlike PathOPEN, filtered PathVQA has **no wrong answers** (Benchmark 2 does not
apply) and **one question per row**, tagged by `question_type`:
- `close-ended` -> Benchmark 3 (Visual Grounding/Reasoning)
- `open-what` / `open-how` / `open-why` / `open-where` / `open-when` -> Benchmark 1
  (Knowledge Interpretation/Deduction, Visual Grounding)

Output CSVs are written in the same column schema as
`data_evaluation/pathologists/scoring_analysis/input/evaluatorN/pathvqa_eval_data.csv`
so they drop directly into the existing `pathvqa_scoring_analysis_individual.ipynb`
as additional synthetic evaluators.

Images for this subset already live locally (unlike PathOPEN core images, which are
resolved via `processed/data.json`) at
`data_evaluation/pathologists/subsets_processing_output/images/pathvqa/`, one file
per `image_id`.

## Checkpointing

Each row here needs only **one** judge call (there's a single question per row,
unlike PathOPEN's up to 11), so the checkpoint key is simply the row's `row_uid`.
Every completed call is appended to
`checkpoints/{model_key}_pathvqa_core.jsonl` immediately; re-running the scoring
cell after an interruption skips whatever `row_uid`s are already checkpointed and
only scores what's missing. A single failed call is logged and skipped for that
pass (not fatal to the whole run) and will be retried automatically next time.


In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())  # so gpu_allocation/judge_models resolve when the CWD is this dir
from gpu_allocation import cuda_visible_devices_for, describe_allocation, max_memory_for

# Which judge(s) this kernel will load. Declared HERE, before torch touches CUDA,
# because CUDA_VISIBLE_DEVICES has no effect once CUDA is initialized - if a torch CUDA
# op has already run in this kernel, restart it.
#
# Both judges run unquantized at bf16 (~64 GB Qwen / ~76 GB InternVL), so each is
# sharded over 3 of the 47.4 GiB A6000s. They are placed in different NUMA islands
# (Qwen 0-2, InternVL 4-6) so they can run concurrently without sharing a PCIe switch.
MODELS_TO_RUN = ["qwenvl"]

os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices_for(*MODELS_TO_RUN)
print(describe_allocation())

In [ ]:
import glob
import os

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from checkpoint import JudgeCheckpoint
from judge_models import JudgeModel
from parallel_judges import run_judges_in_parallel
from prompts.benchmarks import (
    BENCHMARK_1,
    BENCHMARK_3,
    build_benchmark_1_prompt,
    build_benchmark_3_prompt,
)


In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
SUBSETS_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "subsets_processing_output", "data"
)
IMAGES_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "subsets_processing_output", "images", "pathvqa"
)
OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
CHECKPOINT_DIR = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

REPO_ROOT, SUBSETS_DIR, IMAGES_DIR, CHECKPOINT_DIR


In [ ]:
_IMAGE_FILES_BY_ID = {}
for _fname in os.listdir(IMAGES_DIR):
    _stem = _fname.rsplit(".", 1)[0]
    _IMAGE_FILES_BY_ID[_stem] = _fname

len(_IMAGE_FILES_BY_ID)


In [ ]:
def load_pathvqa_image(image_id: str) -> Image.Image:
    fname = _IMAGE_FILES_BY_ID[image_id]
    return Image.open(os.path.join(IMAGES_DIR, fname)).convert("RGB")


## Load judge models

In [ ]:
# MODELS_TO_RUN is set in the first cell (it has to be, to pick GPUs before CUDA init).
# max_memory_for(...) returns LOGICAL device ids - CUDA_VISIBLE_DEVICES renumbers cards,
# so with "4,5,6" visible torch sees 0,1,2.
judges = {
    key: JudgeModel(key, max_memory=max_memory_for(key, MODELS_TO_RUN))
    for key in MODELS_TO_RUN
}

# Verify each judge loaded as intended BEFORE committing to a multi-hour run.
# device_map="auto" silently offloads to CPU/disk when a model does not fit, which turns
# a long run into a multi-day one - catch it here, not hours in.
import collections

import torch

for key, judge in judges.items():
    devs = collections.Counter(str(p.device) for p in judge.model.parameters())
    offloaded = [d for d in devs if d in ("cpu", "meta", "disk")]
    print(f"{key}: devices={dict(devs)}")
    print(f"    system_role={judge.use_system_role}  images_in_template={judge.images_in_template}"
          f"  thinking={'<think>' in judge.system_prompt}")
    print(f"    OFFLOAD -> {offloaded if offloaded else 'none (good)'}")
    if offloaded:
        raise RuntimeError(f"{key} offloaded to {offloaded} - lower max_memory or add a card")

judges

## Score the filtered PathVQA subset

Iterates all `pathvqa_part{1-5}.csv` files (the full pathologist-facing subset, not
a single evaluator's slice).

**Note on join keys**: unlike PathOPEN, `image_id` is **not unique** in filtered
PathVQA (the same image can host several different questions - verified separately,
224 duplicate `Image_ID` values across the pooled human ratings). Each
`evaluatorN/pathvqa_eval_data.csv` is row-for-row aligned with
`subsets_processing_output/data/pathvqa_partN.csv` (verified: identical question
text at every row position, per evaluator). So a global `row_uid` is reconstructed
here by concatenating `pathvqa_part1`..`part5` in sorted order, matching how the
human evaluator CSVs were generated - and this `row_uid` also doubles as the
checkpoint key.

In [ ]:
parts = sorted(glob.glob(os.path.join(SUBSETS_DIR, "pathvqa_part*.csv")))
pathvqa_df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
pathvqa_df["row_uid"] = range(len(pathvqa_df))
len(pathvqa_df), parts

In [ ]:
OPEN_ENDED_TYPES = {"open-what", "open-how", "open-why", "open-where", "open-when"}


def build_pathvqa_subtask(row: pd.Series):
    """Returns (prompt, criteria, key_map) for this row's single question."""
    if row["question_type"] == "close-ended":
        return (
            build_benchmark_3_prompt(row["question"], row["answer"]),
            list(BENCHMARK_3["criteria"].keys()),
            {"Visual Grounding/Reasoning": 'Evaluation CE_Correct_Answer\n(Benchmark 3)'},
        )
    elif row["question_type"] in OPEN_ENDED_TYPES:
        return (
            build_benchmark_1_prompt(row["question"], row["answer"]),
            list(BENCHMARK_1["criteria"].keys()),
            {
                "Knowledge Interpretation/Deduction": 'Evaluation OE_Correct_Answer\n(Benchmark 1)',
                "Visual Grounding": "OE_Correct_Answer_VisGround",
            },
        )
    else:
        raise ValueError(f"Unexpected question_type: {row['question_type']!r}")


In [ ]:
def run_pathvqa_scoring(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathvqa_core.jsonl"))
    n_scored = n_skipped = n_failed = 0

    for _, row in tqdm(pathvqa_df.iterrows(), total=len(pathvqa_df), desc=f"{model_key} pathvqa"):
        item_id = int(row["row_uid"])
        print(f"[{n_scored}/{len(pathvqa_df)}][checkpoint] {model_key} pathvqa: scoring item_id={item_id!r}")
        if checkpoint.is_done(item_id):
            n_skipped += 1
            continue
        prompt, criteria, key_map = build_pathvqa_subtask(row)
        try:
            image = load_pathvqa_image(row["image_id"])
            scores, raw = judge.score(image, prompt, criteria)
        except Exception as e:
            n_failed += 1
            print(f"[{n_scored}/{len(pathvqa_df)}][checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
            continue
        checkpoint.append({
            "item_id": item_id,
            "image_id": row["image_id"],
            "link": row["link"],
            "question_type": row["question_type"],
            "scores": scores,
            "key_map": key_map,
            "raw_response": raw,
        })
        n_scored += 1
        print(f"[{n_scored}/{len(pathvqa_df)}][checkpoint] {model_key} pathvqa: scored item_id={item_id!r} - {scores}")
        
    print(f"{model_key} pathvqa: scored {n_scored} new, {n_skipped} already done, {n_failed} failed this run")
    return checkpoint


# Judges run concurrently - separate GPUs, separate checkpoint files. See parallel_judges.py.
pathvqa_checkpoints = run_judges_in_parallel(run_pathvqa_scoring, judges, MODELS_TO_RUN)


## Post-process: assemble the checkpoint into the evaluator-schema CSV

Reads `checkpoints/{model_key}_pathvqa_core.jsonl` directly from disk (via a fresh
`JudgeCheckpoint(path)`, not the in-memory `pathvqa_checkpoints` object from the
scoring cell above), then joins back against `pathvqa_df` (for the question/answer
text columns) by `row_uid`. Safe to re-run any time `pathvqa_df` is in scope
(re-run the "Score the filtered PathVQA subset" data cell if starting fresh - it's
instant, no GPU involved), without needing the judge models loaded or the scoring
cell to have run in this session.

In [ ]:
def assemble_pathvqa_csv(model_key: str, source_df: pd.DataFrame) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_pathvqa_core.jsonl directly from disk - does
    NOT depend on the `pathvqa_checkpoints` dict from the scoring cell, so this is
    safe to run standalone in a fresh kernel."""
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_pathvqa_core.jsonl"))
    text_by_uid = source_df.set_index("row_uid")[["image_id", "link", "question", "answer", "question_type"]].to_dict(orient="index")

    records = []
    for record in checkpoint.load_all():
        row_uid = record["item_id"]
        text = text_by_uid.get(row_uid, {})
        out_row = {
            "CASE_ID": text.get("image_id", record["image_id"]),
            "Image_ID": text.get("image_id", record["image_id"]),
            "Image_URL": text.get("link", record["link"]),
        }
        if record["question_type"] == "close-ended":
            out_row["CE_Question"] = text.get("question")
            out_row["CE_Correct_Answer"] = text.get("answer")
        else:
            out_row["OE_Question"] = text.get("question")
            out_row["OE_Correct_Answer"] = text.get("answer")
        for criterion, field_name in record["key_map"].items():
            out_row[field_name] = record["scores"].get(criterion)
        records.append(out_row)

    out_df = pd.DataFrame.from_records(records)
    column_order = [
        "CASE_ID", "Image_ID", "Image_URL",
        "OE_Question", "OE_Correct_Answer",
        'Evaluation OE_Correct_Answer\n(Benchmark 1)', "OE_Correct_Answer_VisGround",
        "CE_Question", "CE_Correct_Answer",
        'Evaluation CE_Correct_Answer\n(Benchmark 3)',
    ]
    return out_df.reindex(columns=column_order)


for model_key in MODELS_TO_RUN:
    out_df = assemble_pathvqa_csv(model_key, pathvqa_df)
    out_path = os.path.join(OUTPUT_DIR, f"evaluator_{model_key}", "pathvqa_eval_data.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(model_key, "->", out_path, out_df.shape)


## Next steps

Copy `judge_output/evaluator_internvl/pathvqa_eval_data.csv` and
`judge_output/evaluator_qwenvl/pathvqa_eval_data.csv` into
`data_evaluation/pathologists/scoring_analysis/input/` alongside `evaluator{1-5}/`
and run through `pathvqa_scoring_analysis_individual.ipynb` unchanged.

Raw checkpoint files (`checkpoints/*.jsonl`) retain every model response and
`row_uid` and are never overwritten - they remain the source of truth if the CSV
assembly step ever needs re-running with a different column mapping.

For judge-vs-pathologist agreement, see `judge_pathologist_agreement.ipynb`.